In [0]:
# -------------------------------------------------------------------------
# SCRIPT: 3_taxa_presenca_calc
# LOCAL: 3_gold/src/2_calendario_eventos/
# OBJETIVO: Calcular a taxa de presença por deputado e tipo de evento
# ENTREGA: Taxa de presença por deputado e por tipo de evento ao longo do ano
# ----------------------------------------------------------------------------------------

from pyspark.sql.functions import col, count, round, when

# Carregar tabelas
df_presencas = spark.table("silver_presenca_eventos")
df_fato = spark.table("gold_fato_eventos")

# Obter uma lista única de deputados para o join
df_lista_deputados = df_presencas.select(
    col("id").alias("id_deputado"), 
    col("nome").alias("nome_deputado")
).distinct()

# Presenças confirmadas por deputado e tipo de evento
df_contagem_presenca = df_presencas.groupBy(
    col("id").alias("id_deputado"), 
    col("tipo_evento")
).agg(count("id_evento").alias("presencas_confirmadas"))

# Total de eventos que existiram por tipo na Fato
df_total_eventos_tipo = df_fato.groupBy("descricaoTipo") \
    .agg(count("id_evento").alias("total_eventos_tipo")) \
    .withColumnRenamed("descricaoTipo", "tipo_evento")

df_unificado = df_lista_deputados.join(df_contagem_presenca, "id_deputado", "left")

# Cruzar com o total de eventos por tipo
df_taxa_final = df_unificado.join(df_total_eventos_tipo, "tipo_evento", "inner")

# Cálculo da porcentagem
df_taxa_final = df_taxa_final.withColumn(
    "taxa_presenca_perc", 
    round((col("presencas_confirmadas") / col("total_eventos_tipo")) * 100, 1)
)

# Salvar o resultado final na camada Gold
df_taxa_final.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_taxa_presenca_deputado")

#display(df_taxa_final.orderBy(col("taxa_presenca_perc").asc()))

display(df_taxa_final.orderBy(col("taxa_presenca_perc").desc()))

In [0]:

display(spark.table("gold_fato_eventos").groupBy("descricao").count().orderBy(col("count").desc()))

In [0]:
display(spark.table("gold_fato_eventos").groupBy("descricao").count())